In [36]:
import io
import os
import re
import base64
import google.genai as genai

from datetime import datetime
from google.genai import types
from PIL import Image

In [37]:
from google.colab import userdata
from IPython.display import display, Markdown, HTML, clear_output

In [38]:
#Setting Gemini-Client and Models
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

In [39]:
CHAT_MODEL = "gemini-3.5-flash"
IMAGE_MODEL = "gemini-3.1-flash-lite-image"

client = genai.Client(api_key=GEMINI_API_KEY)

In [40]:
#Core Chatbot Class
#The class manages history, intent detection, and routing to the correct model
class GeminiChatbot:
   # Keywords that signal the user wants an image
  IMAGE_TRIGGERS = [
      "generate an image", "create an image", "draw", "make a pciture",
      "generate a picture", "create a picture", "make an image",
      "show me an image", "generate image", "create image",
      "paint", "illustrate", "sketch", "design an image",
      "visualize", "render an image", "make art", "create art",
      "/image" # Explicit slash command
  ]
  DEFAULT_PROMPT = (
      "You're a helpful, creative, and friendly AI assistant."
      "You remember the full conversation and use that context to"
      "give relevant, coherent answers. Be concise but thorough."
  )

  def __init__(self,
               client:genai.Client,
               chat_model:str=CHAT_MODEL,
               image_model:str=IMAGE_MODEL,
               system_prompt:str|None=None,
               max_history:int=50) -> None:

               self.client = client
               self.chat_model = chat_model
               self.image_model = image_model
               self.max_history = max_history
               self.system_prompt = system_prompt or GeminiChatbot.DEFAULT_PROMPT

               self.history: list[dict] = []
               self.generated_images: list[Image.Image] = []

#Intent Detection
def _is_image_request(self, text:str)->bool:
  lower = text.lower().strip()
  return any(trigger in lower for trigger in self.IMAGE_TRIGGERS)

#History Helpers
def _trim_history(self):
  if len(self.history) > self.max_history:
    self.history = self.history[-self.max_history:]

def _add_to_history(self, role:str, text:str):
  self.history.append({"role":role, "parts":[{"text":text}]})
  self._trim_history()

def get_history_summary(self)->str:
  lines=[]
  for msg in self.history:
    role = "You" if msg["role"] == "user" else "Gemini"
    text = msg["parts"][0]["text"][:120]
    lines.append(f"{role}:{text}")
  return "\n".join(lines) if lines else "(No history yet)"

#Text chat
def _chat(self, user_message:str)->str:
  self._add_to_history("user", user_message)

  response = self.client_models.generate_content(
      model=self.chat_model,
      contents=self.history,
      config=types.GenerateContentConfig(
          system_instructions=self.system_prompt,
          temperature=0.8,
          max_output_tokens=2048
      )
  )

  reply = response.text or ""
  self._add_to_history("model", reply)
  return reply

# Image Generation
def generate_image(self, prompt:str)->tuple[str, Image.Image | None]:
  #Build a rich prompt using conversation context
  context_hint = ""
  if self.history:
    recent = [m['parts'][0]['text'] for m in self.history[-6:]]
    context_hint = (
        "Conversation context for reference:\n" +
        "\n".join(recent) + "\n\n"
    )
  full_prompt = (
      f"{context_hint}",
      f"Generate an image for the following request: {prompt}"
  )
  self._add_to_history

  try:
    response = self.client_models.generate_content(
        model=self.image_model,
        contents=full_prompt,
        config=types.GenerateContentConfig(
            response_modalities=["TEXT", "IMAGE"],
            temperature=1.0,
            max_output_tokens=4096,
        )
    )

    text_reply = ""
    image_out = None

    for part in response.candidates[0].content.parts:
      if part.text:
        text_reply += part.text
      elif part.inline_data:
        image_bytes = part.inline_data.data
        image_out = Image.open(io.BytesIO(image_bytes))
        self.generated_images.append(image_out)

    if not text_reply:
      text_reply = "Here's the generated image: "

    self._add_to_history("model", text_reply + " [image generated]")
    return text_reply, image_out
  except Exception as e:
    error_msg = f"Image generation failed: {e}"
    self._add_to_history("model", error_msg)
    return error_msg, None

#Main Entry Point
def send(self, message:str)->dict:
  if self._is_image_request(message):
    text, image = self.generate_image(message)
    return {"type": "image", "text": text, "image": image}
  else:
    reply = self._chat(message)
    return {"type": "image", "text": reply, "image": image}

def reset(self):
  self.history.clear()
  self.generate_images.clear()


In [41]:
#Gradio Chat UI
import gradio as gr
import tempfile

In [42]:
#Bot instance for the Gradio UI
gradio_bot = GeminiChatbot(client)

In [43]:
def respond(message:str, chat_history:list):
    result = gradio_bot.send(message)
    bot_reply = result["text"]
    image_path = None
    if result["image"]:
        # Save to temp file for gradio display
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        result["image"].save(tmp.name)
        image_path = tmp.name
        bot_reply += f"\n\n![Generated Image] ({image_path})"
    # The `chat_history` input is a list of lists from the `gr.Chatbot`
    chat_history.append([message, bot_reply])

    output_for_chatbot = []
    for user_msg, bot_msg in chat_history:
        if user_msg is not None:
            output_for_chatbot.append({"role": "user", "content": user_msg})
        if bot_msg is not None:
            output_for_chatbot.append({"role": "assistant", "content": bot_msg})
    return "", output_for_chatbot, image_path

In [44]:
with gr.Blocks(title="Gemini Chatbot") as demo:
  gr.Markdown("# Gemini Context-Aware Chatbot")
  gr.Markdown(
      "Chat naturally or use **`/image <prompt>`** to generate images"
      "The bot will remember your whole conversation"
  )

  with gr.Row():
    with gr.Column(scale=3):
      chatbot = gr.Chatbot(height=500, label="Conversation")
      msg = gr.Textbox(placeholder="Type a message: (use /image to generate images)", label="Your Message", lines=1)
      with gr.Row():
        send_btn = gr.Button("Send")
        clear_btn = gr.Button("Clear")

    with gr.Column(scale=1):
      img_output = gr.Image(label="Latest Generated Image", height=300)

  #Wiring Events
  send_btn.click(respond, [msg, chatbot], [msg, chatbot, img_output])
  msg.submit(respond, [msg, chatbot], [msg, chatbot, img_output])
  clear_btn.click(clear_chat, outputs=[chatbot, img_output])

In [ ]:
demo.launch(debug=True, share=True, theme=gr.themes.Ocean())

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e901696a98743d15b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 393, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2280, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1657, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 65, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^